Codice per i dati delle nascite! Almeno lo avete, poi lo riguardo e lo rirendo funzionante. Ne metto uno solo tanto avete visto è uguale per tutte le nazionalità.


In [ ]:
import csv

def leggi_file(percorso):
    with open(percorso, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f, delimiter=";"))

def per_comune(righe, stato):
    stato = stato.strip().lower #toglie spazi
    risultato = {} #fa dizionario da riempire alla fine
    for r in righe: #ogni r è un dizionario
        if r["Stato di nascita"].strip() != stato:
            continue
        codice = r["Codice Istat"] #codice ISTAT del comune è univoco, denominazione potrebbe ripetersi
        if codice not in risultato:
            risultato[codice] = {"comune": r["Denominazione"], "maschi": 0, "femmine": 0, "totale": 0}
        risultato[codice]["maschi"] += int(r["Maschi"])
        risultato[codice]["femmine"] += int(r["Femmine"])
        risultato[codice]["totale"] += int(r["Totale"]) #converte i valori da testo a numero e li somma al contatore, += perchè lo stesso comune può comparire in più righe per lo stesso stato
    return risultato
# il risultato che restituisce è un dizionario completo, nella forma {"001001":{"comune":"Agliè", "maschi": 2, "femmine": 3, "totale": 3}, ...}
def stampa(risultato):
    ordinati = sorted(risultato.items(), key=lambda x: x[1]["totale"], reverse=True) #risultato.items dà coppie (codice, dati), così le posso ordinare. sorted le ordina in base a x[1]["totale"], cioè il totale del comune e reverse = True mette prima i più alti
    for codice, d in ordinati:
        print(f'{d["comune"]} ({codice}): {d["totale"]}  (M {d["maschi"]}, F {d["femmine"]})') #per ogni comune stampa riga tipo Torino (001272): 120 (M60, F60)
    print("Totale generale:", sum(d["totale"] for d in risultato.values())) #numero complessivo di persone di quello stato

def salva(risultato, percorso):
    ordinati = sorted(risultato.items(), key=lambda x: x[1]["totale"], reverse=True)
    with open(percorso, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, delimiter=";")
        w.writerow(["Codice Istat", "Comune", "Maschi", "Femmine", "Totale"])
        for codice, d in ordinati:
            w.writerow([codice, d["comune"], d["maschi"], d["femmine"], d["totale"]])

righe = leggi_file("Dati_nascita_2014.csv")

Qua c'è ancora quella brutta lambda perchè non ho avuto modo di lavorare su questo progetto finora, guardo anche di questa cosa qui e magari la sistemiamo/capiamo

In [ ]:
siria_2014 = per_comune(righe, "Siria")
stampa(siria_2014)
salva(siria_2014, "siria_per_comune_2014.csv")


Questo è il codice per le medie con cui poi fare i confronti. Anche qui vale la stessa logica per cui sono gli stessi elementi del codice ma cambia solo ciò che si vuole analizzare. Il problema qua era capire come mai calcola con questi maggiori o minori, ma rientra tra le cose che devo ancora vedere :').

In [ ]:

FILES = {
    "2014": "Dati_nascita_2014.csv",   
    "2015": "Dati_nascita_2015.csv",
    "2016": "Dati_nascita_2016.csv",
}

ANNI = ["2014", "2015", "2016"]
STATI = ["Siria", "Afghanistan", "Iraq", "Sudan", "Libia"]   

def totale_stato_anno(percorso, stato, anno):
    stato = stato.strip().lower()
    totale = 0
    with open(percorso, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f, delimiter=";"):
            if r["Anno"].strip() == anno and r["Stato di nascita"].strip().lower() == stato:
                totale += int(r["Totale"])
    return totale

print(f'{"Stato":<15}{"2014":>8}{"2015":>8}{"2016":>8}{"Media diff/anno":>18}{"Media var %/anno":>19}')

medie_variazioni = {}
for stato in STATI:
    valori = [totale_stato_anno(FILES[a], stato, a) for a in ANNI]
    diffs = [valori[i + 1] - valori[i] for i in range(len(valori) - 1)]
    percs = [
        (valori[i + 1] - valori[i]) / valori[i] * 100
        for i in range(len(valori) - 1)
        if valori[i] > 0
    ]
    media_diff = sum(diffs) / len(diffs)
    media_perc = sum(percs) / len(percs) if percs else None
    medie_variazioni[stato] = {"valori": valori, "media_diff": media_diff, "media_perc": media_perc}

    perc_txt = f"{media_perc:.1f}" if media_perc is not None else "n.d."
    print(f"{stato:<15}{valori[0]:>8}{valori[1]:>8}{valori[2]:>8}{media_diff:>+18.1f}{perc_txt:>19}")